In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import h5py
from scipy.fft import fftn, ifftn, fftfreq
import illustris_python as il
from matplotlib.colors import LogNorm
import os

In [ ]:
zs = [0.0,0.1,0.2,0.3,0.4,0.5,0.7,1.0,1.5,2.0,3.01,4.01,5.0,6.01,7.01,8.01,9.0,10.0]
ids = [99,91,84,78,72,67,59,50,40,33,25,21,17,13,11,8,6,4]

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
})


In [ ]:
norm_dens = []
voids = []
filaments = []
nodes = []
walls = []
dens = []

with h5py.File('nexus_outputs.h5', 'r') as f:
    for red in zs:
        norm_dens.append(np.transpose(f[f'z_{red}']['normalized_density'][:], (1,2,0)))
        dens.append(np.transpose(f[f'z_{red}']['density'][:], (1,2,0)))
        voids.append(np.transpose(f[f'z_{red}']['voids'][:], (1,2,0)))
        filaments.append(np.transpose(f[f'z_{red}']['filaments'][:], (1,2,0)))
        nodes.append(np.transpose(f[f'z_{red}']['nodes'][:], (1,2,0)))
        walls.append(np.transpose(f[f'z_{red}']['walls'][:], (1,2,0)))
        print(f"Suessfully Loaded Stuff for z={red}")

n = len(norm_dens)
print(n)

In [ ]:
basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"

L = 35.0 # cMpc/h
res = 128

dl  = L / res
print (dl)

In [ ]:
index = 0
slice_index = 67

slic = norm_dens[index]
iidd = ids[index]

z_min = slice_index * dl
z_max = (slice_index + 1) * dl

coords = il.snapshot.loadSubset(basePath, iidd, 'dm', fields = ['Coordinates'])/1000 #in cMpc/h
mask = (coords[:,2] > z_min) & (coords[:,2] < z_max)

x = coords[:, 0][mask]
y = coords[:, 1][mask]
z = coords[:, 2][mask]

fig = plt.figure(figsize=(10,4))

ax1 = fig.add_subplot(1,2,1)
im1 = ax1.imshow(np.log10(slic[:,:,slice_index]),origin = "lower",cmap="inferno", extent = [0,35,0,35])
ax1.set_xlabel("x [cMpc/h]", fontsize = 12)
ax1.set_ylabel("y [cMpc/h]", fontsize = 12)
ax1.set_aspect('equal')
fig.colorbar(im1, ax = ax1, label = r"$\log(1+\delta)$")

ax2 = fig.add_subplot(1,2,2)
counts, xedges, yedges, im2 = ax2.hist2d(x, y, bins = 500, cmap = "inferno", norm = LogNorm())
ax2.set_xlabel("x [cMpc/h]", fontsize = 12)
ax2.set_ylabel("y [cMpc/h]", fontsize = 12)
ax2.set_aspect('equal')
fig.colorbar(im2, ax = ax2, label = "Counts")

fig.suptitle(f"Slice {slice_index} at redshift {zs[index]}", fontsize = 16)

plt.tight_layout();

plt.savefig("git_thing/Img/ps_dtfe_comp.png", transparent = True)
os.system("cd /Users/users/roana/roana/BSc_Thesis/git_thing/Img && git pull origin master && git add . && git commit -m 'Auto-update from python script' && git push origin master")

In [ ]:
vol_n = []
vol_f = []
vol_w = []
vol_v = []

tot_vol = res ** 3

for i in range(n):
    print(i)
    vol_v.append(np.sum(voids[i].flat)/tot_vol)
    vol_w.append(np.sum(walls[i].flat)/tot_vol)
    vol_f.append(np.sum(filaments[i].flat)/tot_vol)
    vol_n.append(np.sum(nodes[i].flat)/tot_vol)

In [ ]:
fig = plt.figure()

ax = fig.add_subplot(111)

ax.plot(zs, vol_n,c="r", label = "nodes", marker = "x")
ax.plot(zs, vol_f,c="orange", label = "filaments", marker = "o")
ax.plot(zs, vol_w,c="g", label = "walls", marker = "*")
ax.plot(zs, vol_v,c="b", label = "voids", marker = "s")

ax.grid()
ax.set_xlabel("Redshift z")
ax.set_ylabel("Volume Fraction")
ax.legend();

In [ ]:
m = 0.0186754872146347 * 10**10 #DM particle mass in Msun/h
Ni = 270
totalMass = m * Ni**3 # total mass contained in simulation box in Msun/h
tot_cells = res**3

In [ ]:
mass_n = []
mass_f = []
mass_w = []
mass_v = []

for i in range(n):
    print(i)
    mass_v.append(np.sum((voids[i] * norm_dens[i]).flat)/tot_cells)
    mass_w.append(np.sum((walls[i] * norm_dens[i]).flat)/tot_cells)
    mass_f.append(np.sum((filaments[i] * norm_dens[i]).flat)/tot_cells)
    mass_n.append(np.sum((nodes[i] * norm_dens[i]).flat)/tot_cells)

In [ ]:
fig = plt.figure()

ax = fig.add_subplot(111)

ax.plot(zs, mass_n,c="r", label = "nodes", marker = "x")
ax.plot(zs, mass_f,c="orange", label = "filaments", marker = "o")
ax.plot(zs, mass_w,c="g", label = "walls", marker = "*")
ax.plot(zs, mass_v,c="b", label = "voids", marker = "s")

ax.grid()
ax.set_xlabel("Redshift z")
ax.set_ylabel("Mass Fraction")
ax.legend();

In [ ]:
mass_n = []
mass_f = []
mass_w = []
mass_v = []

for i in range(n):
    # Multiply by 10**10 to bring dens up to raw Msun/h
    mass_v.append(np.sum((voids[i] * dens[i] * dl**3).flat)/totalMass)
    mass_w.append(np.sum((walls[i] * dens[i] * dl**3).flat)/totalMass)
    mass_f.append(np.sum((filaments[i] * dens[i] * dl**3).flat)/totalMass)
    mass_n.append(np.sum((nodes[i] * dens[i] * dl**3).flat)/totalMass)

In [ ]:
fig = plt.figure()

ax = fig.add_subplot(111)

ax.plot(zs, mass_n,c="r", label = "nodes", marker = "x")
ax.plot(zs, mass_f,c="orange", label = "filaments", marker = "o")
ax.plot(zs, mass_w,c="g", label = "walls", marker = "*")
ax.plot(zs, mass_v,c="b", label = "voids", marker = "s")

ax.grid()
ax.set_xlabel("Redshift z")
ax.set_ylabel("Mass Fraction")
ax.legend();